In [1]:
import pandas as pd
import glob

# load all yearly csv files
files = glob.glob('../data/crime_*.csv')

dfs = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(combined.shape)
print(combined['YEAR'].value_counts().sort_index())

(905669, 17)
YEAR
2015     53597
2016     99430
2017    101338
2018     98888
2019     87184
2020     70894
2021     71721
2022     73852
2023     78043
2024     79087
2025     80964
2026     10671
Name: count, dtype: int64


In [2]:
# clean SHOOTING - Y and 1 both mean shooting
combined['SHOOTING'] = combined['SHOOTING'].apply(
    lambda x: 1 if str(x) in ['1', 'Y'] else 0
)

# drop weird DISTRICT values
combined = combined[~combined['DISTRICT'].isin(['External', 'Outside of'])]
combined = combined[combined['DISTRICT'].notna() & (combined['DISTRICT'] != '')]

# trim whitespace from DAY_OF_WEEK
combined['DAY_OF_WEEK'] = combined['DAY_OF_WEEK'].str.strip()

print(combined.shape)


(899980, 17)


In [3]:
# parse date column
combined['OCCURRED_ON_DATE'] = pd.to_datetime(combined['OCCURRED_ON_DATE'], errors='coerce')

# drop rows with missing key columns
combined = combined.dropna(subset=['MONTH', 'HOUR', 'DAY_OF_WEEK', 'DISTRICT'])

print(combined.shape)
print(combined['YEAR'].value_counts().sort_index())
print("Any NAs:", combined.isnull().sum().sum())

(899980, 17)
YEAR
2015     53468
2016     98947
2017    100805
2018     98206
2019     86066
2020     70370
2021     70586
2022     73612
2023     77766
2024     78786
2025     80731
2026     10637
Name: count, dtype: int64
Any NAs: 1529414


In [4]:
combined.to_csv('../data/boston_crime_combined.csv', index=False)
print("saved!")

saved!
